In [2]:
import pandas as pd

In [2]:
df = pd.read_csv('C:\\Users\\seonu\\Documents\\DR-project\\NEW\\dataset\\서울시_공영주차장_전처리완료.csv', encoding='utf-8-sig')
print(df['주차장종류명'].value_counts())

주차장종류명
노외 주차장    52
노상 주차장    48
Name: count, dtype: int64


- 노상 주차장 제외
- 노외 주차장 중에서 지하주차장, 주차 타워 제외
- 태양광 기설치 주차장 제외

이름으로 구별해보기

In [3]:
nw = df[df['주차장종류명'] == '노외 주차장']
print(nw['주차장명'].tolist())

['가양3동 공영주차장(시)', '가양라이품 공영주차장(시)', '가양역 공영주차장(시)', '개화산역 공영주차장(시)', '개화역 공영주차장(시)', '구로디지털단지역 공영주차장(시)', '구파발역 공영주차장(시)', '남부여성발전센터 공영주차장(시)', '남산예장 관광버스전용 주차장(시)', '남산한옥마을 공영주차장(시)', '당산노외 공영주차장(시)', '대림노외 공영주차장(시)', '도봉산 공영주차장(시)', '도봉산역 공영주차장(시)', '동구로 공영주차장(시)', '동국제강 공영주차장(시)', '동대문 공영주차장(시)', '동작대교 공영주차장(시)', '면목유수지 공영주차장(시)', '명일동 공영주차장(시)', '목동체비지 공영주차장(시)', '방화역 공영주차장(시)', '볕우물 공영주차장(시)', '보라매상업 공영주차장(시)', '복정역 공영주차장(시)', '사당노외 공영주차장(시)', '서울글로벌센터 공영주차장(시)', '석계역 공영주차장(시)', '세종로 공영주차장(시)', '수서역북 공영주차장(시)', '신대방역 공영주차장(시)', '신방화역 공영주차장(시)', '신설동 공영주차장(시)', '신월4동 공영주차장(시)', '신천유수지 공영주차장(시)', '영남 공영주차장(시)', '영등포구청역 공영주차장(시)', '용산주차빌딩 공영주차장(시)', '웃우물 공영주차장(시)', '일원역 공영주차장(시)', '일원터널 공영주차장(시)', '잠실역 공영주차장(시)', '장안1동 공영주차장(시)', '장안2동 공영주차장(시)', '종묘주차장 공영주차장(시)', '창동역(서) 공영주차장(시)', '천왕역 공영주차장(시)', '천호역 공영주차장(시)', '파미에(반포천) 주차장(시)', '학여울 공영주차장(시)', '한강진역 공영주차장(시)', '훈련원공원 공영주차장(시)']


이름만으로는 지하, 주차타워 판별이 어려움

- 로드뷰로 주차유형(지하형/주차빌딩/주차타워/개방형/옥상형/관광버스전용) 확인

- 태양광 기설치 여부 확인

데이터셋: 서울시 공공태양광 시설 현황 

In [14]:
# header가 2번째 행(index=2), 데이터는 3행부터
solar_clean = pd.read_excel(
    r'C:\Users\seonu\Documents\DR-project\NEW\dataset\공공태양광 시설현황_2025_12.xlsx',
    header=2
)

# 필요한 컬럼만 확인
print(solar_clean.shape)
print(solar_clean.columns.tolist()[:10])
print(solar_clean.head(3))

(1356, 16334)
['연번', '설치년도', '관리부서', '개소명', '주소', '설치용량\n[KW]', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9']
   연번  설치년도     관리부서     개소명                        주소  설치용량\n[KW]  \
0   1  2007  양성평등담당관  여성보호센터  강남구 광평로34길 124(수서동 산4-1)        50.0   
1   2  2007    인력개발과   서천연수원         충남 서천군 서면 월호리 621        31.0   
2   3  2007    인재개발원     다솜관          서초구 남부순환로340길 58        55.0   

   Unnamed: 6  Unnamed: 7  Unnamed: 8  Unnamed: 9  ...  Unnamed: 16324  \
0         NaN         NaN         NaN         NaN  ...             NaN   
1         NaN         NaN         NaN         NaN  ...             NaN   
2         NaN         NaN         NaN         NaN  ...             NaN   

   Unnamed: 16325  Unnamed: 16326  Unnamed: 16327  Unnamed: 16328  \
0             NaN             NaN             NaN             NaN   
1             NaN             NaN             NaN             NaN   
2             NaN             NaN             NaN             NaN   

   Unnamed: 16329  Unname

In [15]:
# 필요한 컬럼만 추출
solar_clean = solar_clean[['연번', '설치년도', '관리부서', '개소명', '주소', '설치용량\n[KW]']].copy()
solar_clean.columns = ['연번', '설치년도', '관리부서', '개소명', '주소', '설치용량_KW']

# 빈 행 제거
solar_clean = solar_clean.dropna(subset=['개소명'])

print(solar_clean.shape)
print(solar_clean.head(5))

# 주차장 관련 시설 확인
parking_solar = solar_clean[solar_clean['개소명'].str.contains('주차', na=False)]
print(f"\n주차장 태양광 설치 현황: {len(parking_solar)}개소")
print(parking_solar[['개소명', '주소', '설치년도', '설치용량_KW']].to_string())

(1356, 6)
   연번  설치년도      관리부서            개소명                        주소  설치용량_KW
0   1  2007   양성평등담당관         여성보호센터  강남구 광평로34길 124(수서동 산4-1)     50.0
1   2  2007     인력개발과          서천연수원         충남 서천군 서면 월호리 621     31.0
2   3  2007     인재개발원            다솜관          서초구 남부순환로340길 58     55.0
3   4  2007  동부공원여가센터  서울숲식물원(곤충식물원)               성동구 뚝섬로 273     10.0
4   5  2007    아동복지센터         아동복지센터            강남구 광평로34길 124     50.0

주차장 태양광 설치 현황: 74개소
                                     개소명                              주소  설치년도  설치용량_KW
36                           북서울꿈의숲(주차장)                     강북구 월계로 173  2009    70.00
38                       월드컵공원(노을주차장 주변)                   마포구 상암동 478-1  2009    71.00
40                            평화의주차장 파고라                   마포구 성산동 390-1  2009    66.00
103                      난지물재생센터주차장(고양시)                     덕양구 대덕로 426  2010   100.00
220                            명일동 공영주차장                    강동구 구천면로 395  2011    10.00
252      

서울시 74개 주차장에 태양광 이미 설치됨

우리 주차장 데이터(서울시 공영주차장)랑 비교

In [16]:
# 주차장명 기준으로 매칭
parking = pd.read_csv(r'C:\Users\seonu\Documents\DR-project\NEW\dataset\서울시_공영주차장_전처리완료.csv', encoding='utf-8-sig')

# 태양광 설치된 주차장 이름 목록
solar_parking_names = parking_solar['개소명'].tolist()
print("태양광 설치 주차장 목록:")
for name in solar_parking_names:
    print(f"  {name}")


# 우리 주차장 목록과 비교
print("\n우리 주차장 목록:")
print(parking['주차장명'].tolist())

태양광 설치 주차장 목록:
  북서울꿈의숲(주차장)
  월드컵공원(노을주차장 주변)
  평화의주차장 파고라
  난지물재생센터주차장(고양시)
  명일동 공영주차장
  가로공원주차장
  번동햇살공영주차장
  수유마을시장 공영주차장
  마장축산물시장 서문 공영주차장
  성수2가3동 공영주차장
  장영당길공영주차장
  청량리동공영주차장
  시네마빌딩 공영주차장
  구세군길공영주차장
  송중동공영주차장
  도봉햇빛나눔발전소 1호기(초안산 근린공원주차장)
  남가좌제2동제1공영주차장
  독산4동 공영주차장
  서울시인재개발원 주차장 서초태양광발전소
  강일공영주차장(견인차보관소)
  한남동공영주차장
  뚝섬한강공원 안내센터(차량주차장 지붕)
  신방화역환승주차장
  용답동 공영주차장
  홍익동 공영주차장
  동답공영주차장 
  미아동공영주차장
  냉천길공영주차장
  홍은1동제1공영주차장
  외대역공영주차장
  홍은1동제4공영주차장
  목사랑시장 고객주차장&공유센터
  난곡동제1공영주차장
  암사아리수정수센터(본관동 주차장)
  광암아리수정수센터(주차장)
  구파발역노외주차장
  웃우물노외주차장
  와룡공영주차장
  착한주차안내소
  응봉동 공영주차장
  중산공영주차장
  면목2동공영주차장
  총 14개소, 공영주차장 안내부스
  마곡레포츠센터(주차장)
  보라매제1공영주차장
  신림동공영주차장
  원신공영주차장
  보라매공원정문주차장
  월드컵공원노을공원주차장
  한남유수지공영주차장
  성수2가1동공영주차장
  강변공영주차장
  용답제2공영주차장
  능동공영주차장
  일자산 태양광발전설비(캠핑장 주차장 관리실, 2체육관 주차장)
  중구교육지원센터(동화동 공영주차장)
  도봉햇빛나눔발전소3호기(다락원체육공원주차장)
  신월6동 마을사랑주차장
  목동깨비시장 공유주차장 및 공유센터
  양천구청사 지하주차장
  화곡6-1공영주차장
  고척근린공원 공용주차장
  대명시장 상인회(공영주차장)
  신림동공영주차장
  조원동제1공영주차장
  구로2동 공영주차장
  탄천물재생센터 주

으음 이름이 다라서 텍스트 매칭이 어려움 수동확인 할게욤

In [13]:
parking_solar[['개소명', '주소', '설치년도', '설치용량_KW']].to_csv(
    r'C:\Users\seonu\Documents\DR-project\NEW\OUTPUT\solar_parking_list.csv',
    index=False,
    encoding='utf-8-sig'
)
print("저장 완료")

저장 완료
